# 04 — Preprocessing
Three stages, each with explicit information-loss accounting.

1. **Sensor selection** — drop 6–7 constant sensors (zero variance ⇒ zero loss). 14 kept.
2. **Normalization — the central decision.** FD001/FD003: global z-score; FD002/FD004:
   per-regime z-score (KMeans-6 regimes). *Discipline:* statistics fitted on **healthy cycles
   of fit engines only**, persisted, never refit. Measured stake: +0.4 AUROC on FD004 (nb 08, A1).
3. **Latent smoothing window** w=10 — defined here, consumed by the velocity score; ablated in nb 08 (A2).

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
from src.loaders import prepare, SENSORS, SEEDS, HEALTHY_RUL

In [2]:
# verify regime recovery on FD004: 6 clusters, each op3 value cleanly assigned
d4 = prepare('FD004', seed=42)
t = pd.concat([d4['fit'], d4['cal'], d4['val']])
print('regimes found:', sorted(t.regime.unique()))
print(t.groupby('regime')[['op1', 'op2', 'op3']].mean().round(2))

regimes found: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]
         op1   op2    op3
regime                   
0       42.0  0.84  100.0
1       10.0  0.25  100.0
2       25.0  0.62   60.0
3        0.0  0.00  100.0
4       20.0  0.70  100.0
5       35.0  0.84  100.0


In [3]:
# verify scaler discipline: healthy fit-cycles have ~zero mean / unit std per regime
hf = d4['fit'][d4['fit'].rul > HEALTHY_RUL]
print('healthy fit cycles: mean(|mean_z|) = %.3f, mean(std_z) = %.3f' %
      (hf[SENSORS].mean().abs().mean(), hf[SENSORS].std().mean()))

healthy fit cycles: mean(|mean_z|) = 0.000, mean(std_z) = 1.000


In [4]:
# persist normalized splits for FD001 and FD004 (seed 42) used by notebooks 05-07
for fd in ['FD001', 'FD004']:
    d = prepare(fd, seed=42)
    for k in ['fit', 'cal', 'val']:
        d[k].to_parquet(f'../data/processed/{fd}_seed42_{k}.parquet')
print('normalized splits persisted (other seeds are regenerated on the fly in nb 08).')

normalized splits persisted (other seeds are regenerated on the fly in nb 08).


**Information lost in this pipeline, accounted:** constant sensors (no information by
definition); first w−1 cycles per unit lack smoothed velocity (healthy by construction —
negligible); per-regime statistics assume regime labels are correct (verified above: exact recovery).